In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.opening-checklist",
        description="Evaluates restaurant opening-readiness checklist items from an image and returns a structured JSON output with a calculated readiness score.",
        worker_release="qwen3-instruct",
        text_prompt="""
You are given an image of a restaurant during opening hours.
Your task is to assess a fixed set of opening-readiness checklist items based on what is clearly visible in the image.

           Return ONLY valid JSON.
           Do not include explanation.
           Do not include markdown.
           Do not include commentary.


           ---------------------------------------
           INSTRUCTIONS:


          1. Evaluate each checklist item only based on what is clearly visible.
          2. Do NOT guess or infer the state of an item that is not visible in the frame.
          3. If an item is not visible or cannot be determined, return null for that item.
          4. Use true or false (lowercase, unquoted) for all visible/determinable items. Use boolean values.
          5. Do NOT fabricate or assume readiness based on typical restaurant appearance — base every value strictly on visible evidence.
          6. If the image is not a restaurant interior or exterior, return:
          {
            "document_type": "unknown"
          }
          7. Definitions for each checklist item:
            - "Register_staffed": true if an employee is actively present at a customer-facing register/POS terminal; false if a customer-facing register/POS terminal is visible but unstaffed; null if no customer-facing register/POS terminal (cash drawer, card reader, or customer-facing ordering screen) is visible. Kitchen prep screens and menu boards do not count.
            - "Dining_room_prepared": true if the seating area appears clean, set up, and ready for customers, false if visibly unprepared (dirty tables, chairs stacked, cluttered), null if no dining area is visible.
            - "Exterior_signage_visible": true if exterior signage is visibly illuminated/active, false if signage is visible but off/inactive, null if no exterior signage is in frame.
            - "Menu_board_active": true if a menu board is visibly powered on/displaying content, false if visible but inactive, null if no menu board is in frame.
            - "Beverage_station_stocked": true if the self-serve beverage station appears stocked (cups, ice, syrup/nozzles functional), false if visibly understocked or empty, null if no beverage station is in frame.
            - "Condiment_station_stocked": true if condiment station appears stocked (napkins, utensils, sauces), false if visibly empty or missing items, null if no condiment station is in frame.
            - "Door_unlocked": true if a visible door is confirmed open/unlocked, false if visibly closed/locked, null if no door is in frame.
          8. "Overall_readiness_score" is calculated, not independently judged:
            - Count the number of applicable items (i.e., items with a true or false value, excluding any null items).
            - Count the number of those items marked true.
            - Score = round((true_count / applicable_count) * 10) to the nearest whole number.
            - If fewer than 2 items are applicable (i.e., 6 or more items are null), return null for "Overall_readiness_score" instead of calculating it — there isn't enough visible information to score.


           ---------------------------------------
           RETURN THIS EXACT JSON STRUCTURE:


          {
            "Register_staffed": null,
            "Dining_room_prepared": null,
            "Exterior_signage_visible": null,
            "Menu_board_active": null,
            "Beverage_station_stocked": null,
            "Condiment_station_stocked": null,
            "Door_unlocked": null,
            "Overall_readiness_score": null
          }


           ---------------------------------------

  Return only the JSON object.


        """,
        transform_into=TransformInto(),
       config=InferRuntimeConfig(
           max_new_tokens=200,
           image_size=640
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image

In [ ]:
from pathlib import Path


pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.describe.opening-checklist:latest"
   )
])


with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path("/content/sample_img.jpg") # change to the path of your sample image
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
     print(json.dumps(result, indent=2))

print("Done")
